This notebook trains a biaffine dependency parser on top of the `shekar-ai/albert-base-v2-persian-zwnj-naab-mlm` encoder, using the Persian PerDT treebank from Universal Dependencies.

**Architecture:**
- ALBERT encoder → BiAffine attention for arc prediction + MLP for relation label prediction
- Predicts both **head** (which token is the parent) and **deprel** (dependency relation label)
- Evaluates with **UAS** (Unlabeled Attachment Score) and **LAS** (Labeled Attachment Score)

In [ ]:
!uv pip install conllu transformers accelerate torch onnxscript

In [ ]:
import requests
from conllu import parse

training_set_url = "https://github.com/UniversalDependencies/UD_Persian-PerDT/blob/d728a98d73bd2e52b693c582f249e162a76b6b8f/fa_perdt-ud-train.conllu"
test_set_url = "https://github.com/UniversalDependencies/UD_Persian-PerDT/blob/d728a98d73bd2e52b693c582f249e162a76b6b8f/fa_perdt-ud-test.conllu"
dev_set_url = "https://github.com/UniversalDependencies/UD_Persian-PerDT/blob/d728a98d73bd2e52b693c582f249e162a76b6b8f/fa_perdt-ud-dev.conllu"

dep_relations = [
    "acl", "advcl", "advmod", "amod", "appos", "aux", "case", "cc",
    "ccomp", "compound", "compound:lvc", "conj", "cop", "csubj", "dep",
    "det", "fixed", "flat:name", "flat:num", "goeswith", "iobj", "mark",
    "nmod", "nsubj", "nsubj:pass", "nummod", "obj", "obl", "obl:arg",
    "parataxis", "punct", "root", "vocative", "xcomp"
]

deprel2id = {rel: i for i, rel in enumerate(dep_relations)}
id2deprel = {i: rel for rel, i in deprel2id.items()}

print(f"Number of dependency relations: {len(dep_relations)}")
print(f"Relations: {dep_relations}")

In [ ]:
def get_raw_dataset(url):
    dataset_url = url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
    response = requests.get(dataset_url)
    response.encoding = 'utf-8'
    parsed_sentences = parse(response.text)

    all_words = []
    all_heads = []
    all_deprels = []

    for sentence in parsed_sentences:
        words = []
        heads = []
        deprels = []
        for token in sentence:
            if isinstance(token['id'], tuple):
                continue
            if token['form'] is None or token['head'] is None or token['deprel'] is None:
                continue
            words.append(token['form'])
            heads.append(int(token['head']))
            deprels.append(token['deprel'])
        
        if len(words) > 0:
            all_words.append(words)
            all_heads.append(heads)
            all_deprels.append(deprels)

    return all_words, all_heads, all_deprels

raw_train_words, raw_train_heads, raw_train_deprels = get_raw_dataset(training_set_url)
raw_test_words, raw_test_heads, raw_test_deprels = get_raw_dataset(test_set_url)
raw_dev_words, raw_dev_heads, raw_dev_deprels = get_raw_dataset(dev_set_url)

print(f"Training sentences: {len(raw_train_words)}")
print(f"Dev sentences:      {len(raw_dev_words)}")
print(f"Test sentences:     {len(raw_test_words)}")

all_deprels_in_data = set(d for ds in raw_train_deprels for d in ds)
print(f"\nUnique deprels in training data: {sorted(all_deprels_in_data)}")
missing = all_deprels_in_data - set(dep_relations)
if missing:
    print(f"WARNING: Relations in data but not in label set: {missing}")
    for rel in sorted(missing):
        dep_relations.append(rel)
        deprel2id[rel] = len(deprel2id)
        id2deprel[deprel2id[rel]] = rel
    print(f"Updated to {len(dep_relations)} relations")
else:
    print("All deprels accounted for.")

In [ ]:
idx = 0
print("Words:", raw_train_words[idx])
print("Heads:", raw_train_heads[idx])
print("Deprels:", raw_train_deprels[idx])
print()
for i, (w, h, d) in enumerate(zip(raw_train_words[idx], raw_train_heads[idx], raw_train_deprels[idx]), 1):
    print(f"  {i}\t{w}\t→ head={h}\t{d}")

Dependency parsing is a **word-level** task (each word gets a head pointer to another word), but ALBERT uses subword tokenization. We need to:
1. Tokenize each word into subtokens
2. Track which subtokens belong to which word (word_ids)
3. At inference, pool subtoken representations back to word-level (using the first subtoken of each word)

In [ ]:
from transformers import AlbertTokenizer

model_ckpt = "shekar-ai/albert-base-v2-persian-zwnj-naab-mlm"
tokenizer = AlbertTokenizer.from_pretrained(model_ckpt, use_fast=True, trust_remote_code=True,)

print(f"Tokenizer max length: {tokenizer.model_max_length}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
def tokenize_and_align(words, heads, deprels, tokenizer, max_len=256):
    all_subtoken_ids = []
    word_to_first_subtoken = []
    subtoken_to_word = []

    all_subtoken_ids.append(tokenizer.cls_token_id)
    subtoken_to_word.append(-1)

    for word_idx, word in enumerate(words):
        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if len(sub_ids) == 0:
            sub_ids = [tokenizer.unk_token_id]
        
        word_to_first_subtoken.append(len(all_subtoken_ids))
        for j, sid in enumerate(sub_ids):
            all_subtoken_ids.append(sid)
            subtoken_to_word.append(word_idx)

    all_subtoken_ids.append(tokenizer.sep_token_id)
    subtoken_to_word.append(-1)

    if len(all_subtoken_ids) > max_len:
        all_subtoken_ids = all_subtoken_ids[:max_len]
        subtoken_to_word = subtoken_to_word[:max_len]

    num_words = len(words)
    attention_mask = [1] * len(all_subtoken_ids)

    pad_len = max_len - len(all_subtoken_ids)
    all_subtoken_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    subtoken_to_word += [-1] * pad_len

    head_labels = []
    deprel_labels = []
    for h, d in zip(heads, deprels):
        head_labels.append(h)
        deprel_labels.append(deprel2id.get(d, deprel2id.get('dep', 0)))

    return {
        'input_ids': all_subtoken_ids,
        'attention_mask': attention_mask,
        'word_to_first_subtoken': word_to_first_subtoken,
        'subtoken_to_word': subtoken_to_word,
        'head_labels': head_labels,
        'deprel_labels': deprel_labels,
        'num_words': num_words,
    }

In [ ]:
example = tokenize_and_align(
    raw_train_words[0], raw_train_heads[0], raw_train_deprels[0], tokenizer
)
print(f"Num words: {example['num_words']}")
print(f"Num subtokens (incl special): {sum(example['attention_mask'])}")
print(f"Head labels: {example['head_labels']}")
print(f"Deprel labels: {example['deprel_labels']}")

In [ ]:
import torch
from torch.utils.data import Dataset

class DepParseDataset(Dataset):
    def __init__(self, words_list, heads_list, deprels_list, tokenizer, max_len=256, max_words=128):
        self.data = []
        self.max_words = max_words
        self.max_len = max_len

        skipped = 0
        for words, heads, deprels in zip(words_list, heads_list, deprels_list):
            if len(words) > max_words:
                skipped += 1
                continue
            encoded = tokenize_and_align(words, heads, deprels, tokenizer, max_len)
            self.data.append(encoded)
        
        if skipped > 0:
            print(f"Skipped {skipped} sentences with more than {max_words} words.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        num_words = item['num_words']

        head_labels = item['head_labels'] + [-1] * (self.max_words - num_words)
        deprel_labels = item['deprel_labels'] + [-1] * (self.max_words - num_words)
        
        word_to_first_subtoken = item['word_to_first_subtoken'] + [0] * (self.max_words - num_words)

        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'word_to_first_subtoken': torch.tensor(word_to_first_subtoken, dtype=torch.long),
            'head_labels': torch.tensor(head_labels, dtype=torch.long),
            'deprel_labels': torch.tensor(deprel_labels, dtype=torch.long),
            'num_words': torch.tensor(num_words, dtype=torch.long),
        }

In [ ]:
MAX_LEN = 256
MAX_WORDS = 150

train_dataset = DepParseDataset(raw_train_words, raw_train_heads, raw_train_deprels, tokenizer, MAX_LEN, MAX_WORDS)
dev_dataset = DepParseDataset(raw_dev_words, raw_dev_heads, raw_dev_deprels, tokenizer, MAX_LEN, MAX_WORDS)
test_dataset = DepParseDataset(raw_test_words, raw_test_heads, raw_test_deprels, tokenizer, MAX_LEN, MAX_WORDS)

print(f"Train: {len(train_dataset)}, Dev: {len(dev_dataset)}, Test: {len(test_dataset)}")

Reference: Dozat & Manning (2017) "Deep Biaffine Attention for Neural Dependency Parsing"

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel, AutoConfig
from transformers.modeling_outputs import ModelOutput
from dataclasses import dataclass
from typing import Optional


@dataclass
class DependencyParserOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    arc_logits: Optional[torch.FloatTensor] = None
    rel_logits: Optional[torch.FloatTensor] = None


class BiaffineAttention(nn.Module):
    def __init__(self, in_features_dep, in_features_head, out_features=1):
        super().__init__()
        self.out_features = out_features
        self.W = nn.Parameter(torch.zeros(out_features, in_features_dep, in_features_head))
        self.b_dep = nn.Parameter(torch.zeros(out_features, in_features_dep))
        self.b_head = nn.Parameter(torch.zeros(out_features, in_features_head))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.xavier_uniform_(self.W)

    def forward(self, dep, head):
        batch_size, seq_len, _ = dep.shape
        
        dep_W = torch.einsum('bnd,odh->bonh', dep, self.W)
        bilinear = torch.einsum('bonh,bmh->bonm', dep_W, head)
        
        dep_bias = torch.einsum('bnd,od->bon', dep, self.b_dep)
        head_bias = torch.einsum('bnh,oh->bon', head, self.b_head)
        
        scores = bilinear + dep_bias.unsqueeze(-1) + head_bias.unsqueeze(-2) + self.bias.view(1, -1, 1, 1)
        
        return scores

class BiaffineDependencyParser(nn.Module):
    def __init__(self, encoder_name, num_deprels, arc_hidden=500, rel_hidden=100, dropout=0.33):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden_size = self.encoder.config.hidden_size
        self.num_deprels = num_deprels

        self.root_embedding = nn.Parameter(torch.randn(1, 1, hidden_size))

        self.mlp_arc_dep = nn.Sequential(
            nn.Linear(hidden_size, arc_hidden),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),
        )
        self.mlp_arc_head = nn.Sequential(
            nn.Linear(hidden_size, arc_hidden),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),
        )

        self.mlp_rel_dep = nn.Sequential(
            nn.Linear(hidden_size, rel_hidden),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),
        )
        self.mlp_rel_head = nn.Sequential(
            nn.Linear(hidden_size, rel_hidden),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),
        )

        self.arc_biaffine = BiaffineAttention(arc_hidden, arc_hidden, out_features=1)
        self.rel_biaffine = BiaffineAttention(rel_hidden, rel_hidden, out_features=num_deprels)

        self.dropout = nn.Dropout(dropout)

    def _get_word_embeddings(self, subtoken_embeds, word_to_first_subtoken, num_words, max_words):
        batch_size = subtoken_embeds.size(0)
        hidden_size = subtoken_embeds.size(-1)
        device = subtoken_embeds.device

        word_embeds = torch.zeros(batch_size, max_words, hidden_size, device=device)
        word_mask = torch.zeros(batch_size, max_words, dtype=torch.bool, device=device)

        for b in range(batch_size):
            n = num_words[b].item()
            for w in range(n):
                st_idx = word_to_first_subtoken[b, w].item()
                word_embeds[b, w] = subtoken_embeds[b, st_idx]
            word_mask[b, :n] = True

        root = self.root_embedding.expand(batch_size, 1, hidden_size)
        word_embeds = torch.cat([root, word_embeds], dim=1)
        root_mask = torch.ones(batch_size, 1, dtype=torch.bool, device=device)
        word_mask = torch.cat([root_mask, word_mask], dim=1)

        return word_embeds, word_mask

    def forward(
        self,
        input_ids,
        attention_mask,
        word_to_first_subtoken,
        num_words,
        head_labels=None,
        deprel_labels=None,
    ):
        max_words = word_to_first_subtoken.size(1)

        encoder_output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        subtoken_embeds = self.dropout(encoder_output.last_hidden_state)

        word_embeds, word_mask = self._get_word_embeddings(
            subtoken_embeds, word_to_first_subtoken, num_words, max_words
        )

        arc_dep = self.mlp_arc_dep(word_embeds)
        arc_head = self.mlp_arc_head(word_embeds)
        arc_scores = self.arc_biaffine(arc_dep, arc_head).squeeze(1)

        rel_dep = self.mlp_rel_dep(word_embeds)
        rel_head = self.mlp_rel_head(word_embeds)
        rel_scores = self.rel_biaffine(rel_dep, rel_head)
        rel_scores = rel_scores.permute(0, 2, 3, 1)

        loss = None
        if head_labels is not None and deprel_labels is not None:
            loss = self._compute_loss(arc_scores, rel_scores, head_labels, deprel_labels, num_words)

        return DependencyParserOutput(
            loss=loss,
            arc_logits=arc_scores,
            rel_logits=rel_scores,
        )

    def _compute_loss(self, arc_scores, rel_scores, head_labels, deprel_labels, num_words):
        batch_size = arc_scores.size(0)
        device = arc_scores.device

        arc_loss = 0.0
        rel_loss = 0.0
        total_words = 0

        for b in range(batch_size):
            n = num_words[b].item()

            for i in range(n):
                dep_pos = i + 1
                gold_head = head_labels[b, i].item()
                gold_deprel = deprel_labels[b, i].item()

                if gold_head == -1 or gold_deprel == -1:
                    continue

                arc_logits_i = arc_scores[b, dep_pos, :n+1].unsqueeze(0)
                arc_target = torch.tensor([gold_head], device=device)
                arc_loss += nn.functional.cross_entropy(arc_logits_i, arc_target)

                rel_logits_i = rel_scores[b, dep_pos, gold_head].unsqueeze(0)
                rel_target = torch.tensor([gold_deprel], device=device)
                rel_loss += nn.functional.cross_entropy(rel_logits_i, rel_target)

                total_words += 1

        if total_words > 0:
            arc_loss /= total_words
            rel_loss /= total_words

        return arc_loss + rel_loss

In [ ]:
model = BiaffineDependencyParser(
    encoder_name=model_ckpt,
    num_deprels=len(dep_relations),
    arc_hidden=500,
    rel_hidden=100,
    dropout=0.33,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
import numpy as np

def evaluate_model(model, dataloader, device):
    model.eval()
    correct_arcs = 0
    correct_rels = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                word_to_first_subtoken=batch['word_to_first_subtoken'],
                num_words=batch['num_words'],
            )

            arc_logits = outputs.arc_logits
            rel_logits = outputs.rel_logits

            batch_size = batch['num_words'].size(0)
            for b in range(batch_size):
                n = batch['num_words'][b].item()
                for i in range(n):
                    dep_pos = i + 1
                    gold_head = batch['head_labels'][b, i].item()
                    gold_deprel = batch['deprel_labels'][b, i].item()

                    if gold_head == -1:
                        continue

                    pred_head = arc_logits[b, dep_pos, :n+1].argmax().item()
                    pred_deprel = rel_logits[b, dep_pos, pred_head].argmax().item()

                    if pred_head == gold_head:
                        correct_arcs += 1
                        if pred_deprel == gold_deprel:
                            correct_rels += 1

                    total += 1

    uas = correct_arcs / total if total > 0 else 0
    las = correct_rels / total if total > 0 else 0
    return {'UAS': uas, 'LAS': las, 'total_words': total}

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import time
import os

BATCH_SIZE = 8
LEARNING_RATE = 2e-5
ENCODER_LR = 2e-5
PARSER_LR = 1e-3
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
WARMUP_RATIO = 0.1
SAVE_DIR = "./dep-parser-albert"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = model.to(device)

encoder_params = list(model.encoder.parameters()) + [model.root_embedding]
parser_params = (
    list(model.mlp_arc_dep.parameters()) +
    list(model.mlp_arc_head.parameters()) +
    list(model.mlp_rel_dep.parameters()) +
    list(model.mlp_rel_head.parameters()) +
    list(model.arc_biaffine.parameters()) +
    list(model.rel_biaffine.parameters())
)

optimizer = AdamW([
    {'params': encoder_params, 'lr': ENCODER_LR, 'weight_decay': WEIGHT_DECAY},
    {'params': parser_params, 'lr': PARSER_LR, 'weight_decay': WEIGHT_DECAY},
])

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

In [ ]:
best_las = 0.0
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    num_batches = 0
    start_time = time.time()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            word_to_first_subtoken=batch['word_to_first_subtoken'],
            num_words=batch['num_words'],
            head_labels=batch['head_labels'],
            deprel_labels=batch['deprel_labels'],
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        num_batches += 1

        if (step + 1) % 200 == 0:
            elapsed = time.time() - start_time
            print(f"  Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, "
                  f"Loss: {total_loss/num_batches:.4f}, Time: {elapsed:.1f}s")

    avg_loss = total_loss / num_batches
    epoch_time = time.time() - start_time

    dev_metrics = evaluate_model(model, dev_loader, device)
    
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {avg_loss:.4f} | Time: {epoch_time:.1f}s")
    print(f"  Dev UAS: {dev_metrics['UAS']:.4f} | Dev LAS: {dev_metrics['LAS']:.4f}")

    if dev_metrics['LAS'] > best_las:
        best_las = dev_metrics['LAS']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_las': best_las,
            'dep_relations': dep_relations,
            'deprel2id': deprel2id,
            'id2deprel': id2deprel,
        }, os.path.join(SAVE_DIR, 'best_model.pt'))
        print(f"  ★ New best model saved (LAS: {best_las:.4f})")

print(f"\nTraining complete. Best Dev LAS: {best_las:.4f}")

In [ ]:
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} (Dev LAS: {checkpoint['best_las']:.4f})")

test_metrics = evaluate_model(model, test_loader, device)
print(f"\n=== Test Results ===")
print(f"  UAS: {test_metrics['UAS']:.4f}")
print(f"  LAS: {test_metrics['LAS']:.4f}")
print(f"  Total words evaluated: {test_metrics['total_words']}")

In [ ]:
import torch
import torch.nn as nn

class BiaffineDependencyParserONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder = model.encoder
        self.root_embedding = model.root_embedding
        self.mlp_arc_dep = model.mlp_arc_dep
        self.mlp_arc_head = model.mlp_arc_head
        self.mlp_rel_dep = model.mlp_rel_dep
        self.mlp_rel_head = model.mlp_rel_head
        self.arc_biaffine = model.arc_biaffine
        self.rel_biaffine = model.rel_biaffine

    def forward(self, input_ids, attention_mask, word_to_first_subtoken):
        encoder_output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        subtoken_embeds = encoder_output.last_hidden_state

        batch_size, num_words = word_to_first_subtoken.shape
        hidden_size = subtoken_embeds.size(-1)

        idx = word_to_first_subtoken.unsqueeze(-1).expand(-1, -1, hidden_size)
        word_embeds = torch.gather(subtoken_embeds, 1, idx)

        root = self.root_embedding.expand(batch_size, 1, hidden_size)
        word_embeds = torch.cat([root, word_embeds], dim=1)

        arc_dep = self.mlp_arc_dep(word_embeds)
        arc_head = self.mlp_arc_head(word_embeds)
        arc_scores = self.arc_biaffine(arc_dep, arc_head).squeeze(1)

        rel_dep = self.mlp_rel_dep(word_embeds)
        rel_head = self.mlp_rel_head(word_embeds)
        rel_scores = self.rel_biaffine(rel_dep, rel_head)
        rel_scores = rel_scores.permute(0, 2, 3, 1)

        return arc_scores, rel_scores


print('ONNX wrapper class defined.')

In [ ]:
ONNX_PATH = os.path.join(SAVE_DIR, 'albert_persian_dep_parser.onnx')

checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'), map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.cpu()

onnx_model = BiaffineDependencyParserONNXWrapper(model)
onnx_model.eval()

dummy_words = ['ما', 'کتاب', 'می\u200cخوانیم']
n_dummy = len(dummy_words)
max_len_export = 32

dummy_ids = [tokenizer.cls_token_id]
dummy_w2st = []
for word in dummy_words:
    sub_ids = tokenizer.encode(word, add_special_tokens=False)
    if not sub_ids:
        sub_ids = [tokenizer.unk_token_id]
    dummy_w2st.append(len(dummy_ids))
    dummy_ids.extend(sub_ids)
dummy_ids.append(tokenizer.sep_token_id)
dummy_mask = [1] * len(dummy_ids)
pad_len = max_len_export - len(dummy_ids)
dummy_ids += [tokenizer.pad_token_id] * pad_len
dummy_mask += [0] * pad_len

dummy_input_ids  = torch.tensor([dummy_ids], dtype=torch.long)
dummy_attn_mask  = torch.tensor([dummy_mask], dtype=torch.long)
dummy_w2st_t     = torch.tensor([dummy_w2st], dtype=torch.long)

torch.onnx.export(
    onnx_model,
    (dummy_input_ids, dummy_attn_mask, dummy_w2st_t),
    ONNX_PATH,
    input_names=['input_ids', 'attention_mask', 'word_to_first_subtoken'],
    output_names=['arc_logits', 'rel_logits'],
    dynamic_axes={
        'input_ids':              {0: 'batch_size', 1: 'seq_len'},
        'attention_mask':         {0: 'batch_size', 1: 'seq_len'},
        'word_to_first_subtoken': {0: 'batch_size', 1: 'num_words'},
        'arc_logits':             {0: 'batch_size', 1: 'num_words_plus_1', 2: 'num_words_plus_1'},
        'rel_logits':             {0: 'batch_size', 1: 'num_words_plus_1', 2: 'num_words_plus_1'},
    },
    opset_version=14,
)

print(f'Exported to {ONNX_PATH}')

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

Q8_PATH = os.path.join(SAVE_DIR, 'albert_persian_dep_parser_q8.onnx')

quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=Q8_PATH,
    weight_type=QuantType.QInt8,
)

print(f'Quantized model saved to {Q8_PATH}')

In [ ]:
import onnxruntime
import numpy as np

session = onnxruntime.InferenceSession(Q8_PATH, providers=['CPUExecutionProvider'])

feed = {
    'input_ids':              dummy_input_ids.numpy(),
    'attention_mask':         dummy_attn_mask.numpy(),
    'word_to_first_subtoken': dummy_w2st_t.numpy(),
}

arc_logits, rel_logits = session.run(None, feed)
print(f'arc_logits shape: {arc_logits.shape}')
print(f'rel_logits shape: {rel_logits.shape}')

print('\nParse results for dummy sentence:')
print(f'{"ID":<5} {"Word":<15} {"Head":<8} {"Deprel"}')
print('-' * 45)
for i in range(n_dummy):
    dep_pos = i + 1
    pred_head = int(arc_logits[0, dep_pos, :n_dummy + 1].argmax())
    pred_deprel = int(rel_logits[0, dep_pos, pred_head].argmax())
    head_word = dummy_words[pred_head - 1] if pred_head > 0 else '[ROOT]'
    print(f'{i+1:<5} {dummy_words[i]:<15} {pred_head:<3} ({head_word:<10}) {id2deprel[pred_deprel]}')

print('\nQuantized model validated successfully.')

In [ ]:
def evaluate_per_relation(model, dataloader, device, id2deprel):
    model.eval()
    rel_correct = {}
    rel_total = {}

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                word_to_first_subtoken=batch['word_to_first_subtoken'],
                num_words=batch['num_words'],
            )

            arc_logits = outputs.arc_logits
            rel_logits = outputs.rel_logits

            for b in range(batch['num_words'].size(0)):
                n = batch['num_words'][b].item()
                for i in range(n):
                    dep_pos = i + 1
                    gold_head = batch['head_labels'][b, i].item()
                    gold_deprel = batch['deprel_labels'][b, i].item()
                    if gold_head == -1:
                        continue

                    pred_head = arc_logits[b, dep_pos, :n+1].argmax().item()
                    pred_deprel = rel_logits[b, dep_pos, pred_head].argmax().item()

                    rel_name = id2deprel.get(gold_deprel, '?')
                    rel_total[rel_name] = rel_total.get(rel_name, 0) + 1
                    if pred_head == gold_head and pred_deprel == gold_deprel:
                        rel_correct[rel_name] = rel_correct.get(rel_name, 0) + 1

    print(f"{'Relation':<20} {'Correct':>8} {'Total':>8} {'LAS':>8}")
    print("-" * 48)
    for rel in sorted(rel_total.keys(), key=lambda x: rel_total[x], reverse=True):
        correct = rel_correct.get(rel, 0)
        total = rel_total[rel]
        las = correct / total if total > 0 else 0
        print(f"{rel:<20} {correct:>8} {total:>8} {las:>8.4f}")

print("\n=== Per-Relation LAS on Test Set ===")
evaluate_per_relation(model, test_loader, device, id2deprel)

Parse a new Persian sentence using the trained model.

In [ ]:
def parse_sentence(model, tokenizer, words, device, max_len=256):
    model.eval()
    n = len(words)

    all_subtoken_ids = [tokenizer.cls_token_id]
    word_to_first_subtoken = []

    for word in words:
        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if len(sub_ids) == 0:
            sub_ids = [tokenizer.unk_token_id]
        word_to_first_subtoken.append(len(all_subtoken_ids))
        all_subtoken_ids.extend(sub_ids)

    all_subtoken_ids.append(tokenizer.sep_token_id)

    attention_mask = [1] * len(all_subtoken_ids)
    pad_len = max_len - len(all_subtoken_ids)
    all_subtoken_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len

    max_words = n
    w2st = word_to_first_subtoken + [0] * (max_words - len(word_to_first_subtoken))

    input_ids = torch.tensor([all_subtoken_ids], dtype=torch.long, device=device)
    attn = torch.tensor([attention_mask], dtype=torch.long, device=device)
    w2st_t = torch.tensor([w2st], dtype=torch.long, device=device)
    nw = torch.tensor([n], dtype=torch.long, device=device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attn,
            word_to_first_subtoken=w2st_t,
            num_words=nw,
        )

    arc_logits = outputs.arc_logits[0]
    rel_logits = outputs.rel_logits[0]

    results = []
    for i in range(n):
        dep_pos = i + 1
        pred_head = arc_logits[dep_pos, :n+1].argmax().item()
        pred_deprel = rel_logits[dep_pos, pred_head].argmax().item()
        results.append((pred_head, id2deprel[pred_deprel]))

    return results